# Compartment assignment

Splits the integrated object into the three compartments this project works
with - epithelial, mesenchymal and immune - so each can be re-clustered and
annotated on its own.

Deliberately coarse: `resolution = 0.1`. The goal here is compartments
separated by unambiguous markers, not cell types. Fine structure is resolved
per compartment later, where each one gets a resolution suited to it.

No cells are filtered in this notebook. Quality filtering happens inside each
compartment object, once it is re-clustered finely enough to separate junk from
real populations.

Set `DATA` in `config.py` before running.

In [ ]:
import scanpy as sc

from config import DATA

# every stochastic step below takes random_state explicitly - clustering and
# UMAP are both seeded, so a rerun on the same input reproduces this exactly
SEED = 0

## 1. Load the integrated object

Written by `integration.py`. `X_scVI` is the batch-corrected latent space that
everything downstream is built on.

In [ ]:
adata = sc.read_h5ad(DATA / 'nerandomilast_integrated.h5ad')
print(adata.shape)
adata

## 2. Neighbours and UMAP

Built on `X_scVI`, not PCA - the point of integration was to remove the
per-sample batch effect, and clustering on PCA would put it straight back.

`n_neighbors=30` is higher than the scanpy default of 15. With ~113k cells a
larger neighbourhood gives a smoother graph, which is what coarse clustering
wants; too small and compartments fragment into noise.

In [ ]:
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=30, random_state=SEED)
sc.tl.umap(adata, random_state=SEED)

## 3. Cluster at low resolution

In [ ]:
sc.tl.leiden(adata, resolution=0.1, random_state=SEED)

print(f"{adata.obs['leiden'].nunique()} clusters")
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False)

## 4. Canonical markers

**This is the evidence for the mapping in the next cell.** Each cluster is
called from the marker it expresses, not from where it sits on the UMAP:

| Marker | Compartment |
| --- | --- |
| `Epcam` | Epithelial |
| `Pdgfra`, `Pdgfrb`, `Msln` | Mesenchyme |
| `Ptprc` | Immune |
| `Pecam1` | Endothelial |

Read these plots before trusting the assignment below. A cluster expressing two
of these markers is a doublet cluster and should not be assigned to either.

In [ ]:
sc.pl.umap(
    adata,
    color=['Epcam', 'Pdgfra', 'Pdgfrb', 'Msln', 'Ptprc', 'Pecam1'],
    frameon=False
)

## 5. Map clusters to compartments

The assignment below was made **by reading the marker plots above** - each
cluster was given the compartment whose marker it expressed. It is a record of
that judgement, not something the code derives.

Clusters expressing none of the four markers are left as `other`.

> **These cluster numbers are only valid for the clustering that produced the
> deposited object.** Leiden numbering is not stable: re-running integration, or
> clustering a different set of cells, renumbers everything. If you re-run this
> notebook, **do not trust this dict** - read the marker plots above and rebuild
> it. The markers are the portable instruction; the numbers are a record.

In [ ]:
COMPARTMENTS = {
    'Mesenchyme':  ['0', '4', '8', '10'],
    'Epithelial':  ['3', '9'],
    'Endothelial': ['7', '12'],
    'Immune':      ['1', '2', '5', '6', '11', '13', '14'],
}

adata.obs['compartment'] = 'other'
for name, clusters in COMPARTMENTS.items():
    adata.obs.loc[adata.obs['leiden'].isin(clusters), 'compartment'] = name

adata.obs['compartment'] = adata.obs['compartment'].astype('category')

print(adata.obs['compartment'].value_counts().to_string())

In [ ]:
sc.pl.umap(adata, color='compartment', frameon=False)

Sanity check that the split tracks condition and sample rather than cutting
across them - a compartment that appears in only one sample is a batch effect,
not biology.

In [ ]:
sc.pl.umap(adata, color=['type', 'sample'], frameon=False)

## 6. Write one object per compartment

Three new objects are written, one per compartment.

- **Endothelial** and **other** don't matter to us. This project concerns the epithelial,
  mesenchymal and immune response to nerandomilast

`.copy()` matters here. Slicing an AnnData returns a *view* onto the parent, not
an independent object; writing a view works but anndata warns, and any later
edit to a view raises. Copying makes each compartment a real object.

In [ ]:
CARRIED_FORWARD = ['Epithelial', 'Mesenchyme', 'Immune']

for name in CARRIED_FORWARD:
    sub = adata[adata.obs['compartment'] == name].copy()

    out = DATA / f'{name.lower()}.h5ad'
    sub.write_h5ad(out)
    print(f'{out.name}: {sub.n_obs:,} cells')

The full object, compartment labels included, is kept as the record of where
every cell went - including the endothelial and `other` cells that stop here.

In [ ]:
adata.write_h5ad(DATA / 'nerandomilast_compartments.h5ad')
print(f'{adata.n_obs:,} cells written')